# 分钟级等 beta 资产选择：来源及整股可达性预检

先用逐层点时 f696+b472 原生账户与已验证Qlib双价一分钟来源，在同一完成bar后比较现金增配SOXX和等名义beta增配SOXL的最大共同目标空间。仅检查 t+1可用、t+2填单、双边整股、现金／SOXL结构与20交易日标签成熟日历；不读取未来收益、不拟合、不运行新资金账户。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
import hashlib,json,warnings
from copy import deepcopy
from pathlib import Path
import exchange_calendars as xc
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from src.workflow.minute_action_value import _source
from src.backtest.event_guard_clock import guard_slot
from src.backtest.ibkr_tiered import TieredFees,settle_tiered_event
warnings.filterwarnings("ignore",message="The 'generic' unit for NumPy timedelta")
root=Path.cwd();card=root/"research/stage1_event_beta_asset_choice_support_20261029"
plan_bytes=(card/"plan.json").read_bytes();plan=json.loads(plan_bytes)
assert hashlib.sha256((root/plan["provider_uri"]/"qlib_store_manifest.json").read_bytes()).hexdigest()==plan["provider_manifest_sha256"]
qlib.init(provider_uri=str(root/plan["provider_uri"]),region="us",kernels=1)
source,source_receipt=_source(root,plan["training_source"],["analysis/native/dated_f696_0"])
frame_source,frame_receipt=_source(root,plan["frame_source"],["inputs/covered_frame"])
book=source["analysis/native/dated_f696_0"];frame=frame_source["inputs/covered_frame"].reset_index(drop=True)
assert len(frame)==plan["expected_parent_sessions"]==len(book["authority"])
assert frame.trade_date.equals(book["authority"].trade_date)
assert book["audit"]["status"]=="passed" and book["audit"]["slippage_bps"]==0
bars=D.features(["SOXL","SOXX"],["$open","$close","$raw_open","$price_valid"],start_time=plan["training_start"]+" 09:31:00",end_time=plan["training_end"]+" 16:00:00",freq="1min")
parts=[]
for symbol in ("SOXL","SOXX"):
    item=bars.xs(symbol,level="instrument").copy()
    item.columns=[symbol+"_"+field.removeprefix("$") for field in item]
    parts.append(item)
quotes=pd.concat(parts,axis=1).sort_index()
schedule=xc.get_calendar("XNYS",start=frame.trade_date.min(),end=frame.trade_date.max()).schedule
closes={pd.Timestamp(day).tz_localize(None):pd.Timestamp(row.close).tz_convert("America/New_York").tz_localize(None) for day,row in schedule.iterrows()}
orders=book["orders"].sort_values("stamp")
by_day={pd.Timestamp(day):item for day,item in orders.groupby("trade_date",sort=True)}
fees=TieredFees(plan["tariff"],slippage_bps=0)
perday=[];event_slots=0;target_room_slots=0;both_funded_slots=0
for index,row in frame.iterrows():
    day=pd.Timestamp(row.trade_date);actual=book["authority"].iloc[index]
    current_orders=by_day.get(day,pd.DataFrame())
    asset=quotes.loc[quotes.index.normalize()==day]
    status=("terminal_20_session_label" if index+20>=len(frame) else
            "original_recovery" if bool(actual.recovery_action) else
            "source_invalid" if asset.empty or not asset[["SOXL_price_valid","SOXX_price_valid"]].eq(1).all().all() else "eligible")
    record={"trade_date":str(day.date()),"year":int(day.year),"status":status,"minute_slots":0,"target_room_slots":0,"soxx_whole_buy_slots":0,"soxl_whole_buy_slots":0,"both_whole_buy_slots":0,"max_common_beta_room":0.,"median_positive_common_beta_room":None,"label_maturity_date":str(frame.iloc[index+20].trade_date.date()) if index+20<len(frame) else None}
    if status=="eligible":
        opened=day+pd.Timedelta(hours=9,minutes=47);position=book["native_positions"].get(opened)
        assert position is not None
        q=np.array([position.get_stock_amount(symbol) if position.check_stock(symbol) else 0. for symbol in ("SOXL","SOXX")],float)
        cash=float(position.get_cash())
        close=closes[day]
        eligible=asset.loc[(asset.index>=opened+pd.Timedelta(minutes=1)) & (asset.index+pd.Timedelta(minutes=2)<close)]
        event_slots+=len(eligible);record["minute_slots"]=len(eligible)
        branch=deepcopy(fees)
        for order in current_orders.itertuples(index=False):
            branch.commit(int(order.raw_shares),stamp=order.stamp,cap_applied=bool(order.cap_applied))
        positive_room=[]
        for minute in eligible.itertuples():
            slot=guard_slot(minute.Index,session_open=day+pd.Timedelta(hours=9,minutes=30),session_close=close,original_open_fill=opened)
            assert slot is not None
            mark=np.array([minute.SOXL_close,minute.SOXX_close],float)
            nav=float(q@mark+cash)
            assert np.isfinite(nav) and nav>0
            weights=np.r_[q*mark,cash]/nav
            d=min(max(0.,weights[2]-.15),3*max(0.,.30-weights[0]),max(0.,1-weights[1]))
            if d<=1e-10:continue
            target_room_slots+=1;record["target_room_slots"]+=1
            positive_room.append(float(d));record["max_common_beta_room"]=max(record["max_common_beta_room"],float(d))
            targets={"SOXX":weights+np.array([0.,d,-d]),"SOXL":weights+np.array([d/3,0.,-d/3])}
            assert all(abs(x.sum()-1)<1e-8 and x.min()>=-1e-8 and x[0]<=.30+1e-8 and x[2]>=.15-1e-8 for x in targets.values())
            assert abs((3*targets["SOXX"][0]+targets["SOXX"][1])-(3*targets["SOXL"][0]+targets["SOXL"][1]))<1e-8
            quote=asset.loc[slot.fill_at+pd.Timedelta(minutes=1)]
            fill=np.array([quote.SOXL_open,quote.SOXX_open],float)
            raw=np.array([quote.SOXL_raw_open,quote.SOXX_raw_open],float)
            funded={}
            for symbol,desired in targets.items():
                trade=settle_tiered_event(q,cash,desired=desired,adjusted_prices=fill,raw_prices=raw,stamp=slot.fill_at,fees=deepcopy(branch))
                orders_now=trade["orders"]
                after_nav=float(trade["quantities"]@fill+trade["cash"])
                legal=bool(np.isfinite(after_nav) and after_nav>0 and trade["cash"]/after_nav>=.15-1e-8 and trade["quantities"][0]*fill[0]/after_nav<=.30+1e-8)
                funded[symbol]=bool(legal and len(orders_now)==1 and orders_now[0]["symbol"]==symbol and not orders_now[0]["sell"])
                if funded[symbol]:record[symbol.lower()+"_whole_buy_slots"]+=1
            if funded["SOXL"] and funded["SOXX"]:
                record["both_whole_buy_slots"]+=1;both_funded_slots+=1
        if positive_room:record["median_positive_common_beta_room"]=float(np.median(positive_room))
    perday.append(record)
    for order in current_orders.itertuples(index=False):
        fees.commit(int(order.raw_shares),stamp=order.stamp,cap_applied=bool(order.cap_applied))
assert event_slots<=plan["budget"]["maximum_event_slots"]
perday=pd.DataFrame(perday)
annual=[]
for year,part in perday.groupby("year"):
    both=part.loc[part.both_whole_buy_slots.gt(0)]
    annual.append({"year":int(year),"parent_days":len(part),"source_eligible_days":int(part.status.eq("eligible").sum()),"minute_slots":int(part.minute_slots.sum()),"target_room_slots":int(part.target_room_slots.sum()),"soxx_whole_buy_days":int(part.soxx_whole_buy_slots.gt(0).sum()),"soxl_whole_buy_days":int(part.soxl_whole_buy_slots.gt(0).sum()),"both_whole_buy_days":len(both),"both_whole_buy_slots":int(part.both_whole_buy_slots.sum()),"median_both_day_common_beta_room":float(both.median_positive_common_beta_room.median()) if len(both) else None})
current=[row for row in annual if row["year"]==2022][0]
gate=(sum(x["both_whole_buy_days"] for x in annual)>=plan["support_gate"]["minimum_both_funded_days"] and current["both_whole_buy_days"]>=plan["support_gate"]["minimum_both_funded_2022_days"] and sum(x["both_whole_buy_days"]>=20 for x in annual)>=plan["support_gate"]["minimum_years_with_20_both_days"] and current["median_both_day_common_beta_room"] is not None and current["median_both_day_common_beta_room"]>=plan["support_gate"]["minimum_median_common_beta_room"])
audit={"experiment":plan["experiment"],"plan_sha256":hashlib.sha256(plan_bytes).hexdigest(),"source_recorder_ids":{"parent":plan["training_source"]["recorder_id"],"frame":plan["frame_source"]["recorder_id"]},"source_readback_entries":{"parent":len(source_receipt),"frame":len(frame_receipt)},"parent_sessions":len(frame),"status_counts":perday.status.value_counts().to_dict(),"event_slots":event_slots,"target_room_slots":target_room_slots,"both_funded_slots":both_funded_slots,"annual":annual,"support_gate_pass":bool(gate),"new_model_fits":0,"new_native_accounts":0,"future_outcomes_read":0,"historical_research_passed":False,"decision":"joint_minute_asset_choice_support_pass_needs_episode_replay" if gate else "not_promoted_joint_minute_asset_choice_support_failed"}
(card/"annual_support.csv").write_text(pd.DataFrame(annual).to_csv(index=False))
(card/"perday_source_support.csv").write_text(perday.to_csv(index=False))
(card/"audit.json").write_text(json.dumps(audit,ensure_ascii=False,indent=2)+"\n")
print(json.dumps({"decision":audit["decision"],"event_slots":event_slots,"both_funded_slots":both_funded_slots,"annual":annual},ensure_ascii=False,indent=2))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[28908:MainThread](2026-10-29 11:36:47,963) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[28908:MainThread](2026-10-29 11:36:48,514) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[28908:MainThread](2026-10-29 11:36:48,515) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


{
  "decision": "joint_minute_asset_choice_support_pass_needs_episode_replay",
  "event_slots": 377810,
  "both_funded_slots": 283513,
  "annual": [
    {
      "year": 2018,
      "parent_days": 53,
      "source_eligible_days": 53,
      "minute_slots": 19250,
      "target_room_slots": 19209,
      "soxx_whole_buy_days": 53,
      "soxl_whole_buy_days": 53,
      "both_whole_buy_days": 53,
      "both_whole_buy_slots": 17603,
      "median_both_day_common_beta_room": 0.32896988682275563
    },
    {
      "year": 2019,
      "parent_days": 252,
      "source_eligible_days": 252,
      "minute_slots": 92700,
      "target_room_slots": 75799,
      "soxx_whole_buy_days": 238,
      "soxl_whole_buy_days": 226,
      "both_whole_buy_days": 226,
      "both_whole_buy_slots": 66325,
      "median_both_day_common_beta_room": 0.058470246142406784
    },
    {
      "year": 2020,
      "parent_days": 253,
      "source_eligible_days": 242,
      "minute_slots": 89180,
      "target_room_slot

## 中文结论

逐层点时 f696+b472 1056日与 Qlib 双价一分钟来源核对；排除20个20日终端未知、10个原恢复和1个异常源日后，1025日有377810个合法分钟时钟。两种同名义beta现金买入动作在283513个时点均可分别形成单笔整股买单，2022年支持226日，共同最大beta容量中位数0.551。固定来源支持门通过：`joint_minute_asset_choice_support_pass_needs_episode_replay`。

本卡只证明分钟时钟、来源、整股与20日标签终点可达；0未来收益、0新模型、0新账户。最大目标空间不等于批准的行动幅度或SOXX超额。下一步先实现干预后20日自身状态账户回放，并以无干预分支逐股、现金、费用、NAV和风险状态复现封存父本；若机械门失败即停止。历史分钟首次真实接收仍未认证。详情见 `summary.json`、`annual_support.csv`、`conclusion.md`。